# Class 13 - GroupBy & Aggregation
### Module 3 . Week 5 . Friday

Most analytical questions are really grouping questions:

- *Which product category generates the most revenue?*
- *How does average order value differ by region?*
- *Which city had the highest discount rate last quarter?*

All of these follow the same three-step pattern that Pandas calls
**Split-Apply-Combine**: split the data into groups by some key,
apply a function independently within each group, combine the
results back into a single output.

Today's dataset: 600 real-ish e-commerce orders from 2023 covering
5 product categories across 4 Pakistani regions.

## Learning Objectives

- Understand the Split-Apply-Combine mental model
- Use `.groupby()` with single and multiple keys
- Write custom aggregations with `.agg()`
- Build cross-tabular summaries with `pd.pivot_table()`
- Distinguish `.transform()` from `.agg()` and know when you need each

In [47]:
import pandas as pd
import numpy as np

DATA_DIR = "./datasets"
# pd.set_option("display.float_format", "{:,.1f}".format)
# pd.set_option("display.max_columns", None)

df = pd.read_csv(f"{DATA_DIR}/sales.csv", parse_dates=["date"])
print(f"Loaded {len(df)} orders  |  date range: {df['date'].min().date()} to {df['date'].max().date()}")
df.head()

Loaded 600 orders  |  date range: 2023-01-01 to 2023-12-31


,order_id,date,customer_id,product,category,region,city,quantity,unit_price,discount_pct,revenue
0,ORD10210,2023-01-01,C1098,Shirt,Clothing,East,Lahore,2,2847.0,10,5125.0
1,ORD10496,2023-01-01,C1033,Phone,Electronics,East,Faisalabad,7,20975.0,5,139484.0
2,ORD10173,2023-01-02,C1777,Jeans,Clothing,North,Islamabad,2,4056.0,15,6895.0
3,ORD10081,2023-01-02,C1042,Biology,Books,West,Quetta,5,515.0,5,2446.0
4,ORD10240,2023-01-02,C1880,Jacket,Clothing,North,Islamabad,4,3688.0,0,14752.0


## First Look

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   order_id      600 non-null    object        
 1   date          600 non-null    datetime64[ns]
 2   customer_id   600 non-null    object        
 3   product       600 non-null    object        
 4   category      600 non-null    object        
 5   region        600 non-null    object        
 6   city          600 non-null    object        
 7   quantity      600 non-null    int64         
 8   unit_price    600 non-null    float64       
 9   discount_pct  600 non-null    int64         
 10  revenue       600 non-null    float64       
dtypes: datetime64[ns](1), float64(2), int64(2), object(6)
memory usage: 51.7+ KB


In [4]:
df.describe()

,date,quantity,unit_price,discount_pct,revenue
count,600,600.000000,600.000000,600.000000,600.000000
mean,2023-06-22 12:24:00,3.960000,6455.091667,4.883333,24400.531667
min,2023-01-01 00:00:00,1.000000,241.000000,0.000000,228.000000
25%,2023-03-28 18:00:00,2.000000,654.000000,0.000000,2080.500000
50%,2023-06-17 12:00:00,4.000000,1659.500000,0.000000,5337.000000
75%,2023-09-20 00:00:00,6.000000,3692.750000,10.000000,16048.000000
max,2023-12-31 00:00:00,7.000000,34616.000000,20.000000,242312.000000
std,NaN,1.954849,10106.995698,6.127721,44339.808417


In [9]:
# df['category'].value_counts()

In [10]:
# df['region'].value_counts()

In [8]:
# Quick frequency check on categorical columns
for col in ["category", "region"]:
    print(f"{col}:\n{df[col].value_counts()}\n")

category:
category
Electronics    126
Books          126
Clothing       124
Food           117
Healthcare     107
Name: count, dtype: int64

region:
region
North    163
West     149
South    148
East     140
Name: count, dtype: int64



## The Split-Apply-Combine Pattern

Before writing any code, commit this mental model to memory:

```
Split  : df.groupby("category")   -- divide rows into groups
Apply  : .sum()  .mean()  .agg()  -- compute something per group
Combine: Pandas stitches results into a new Series or DataFrame
```

Every `.groupby()` call you ever write follows this path.

## Single-key GroupBy

In [22]:
# df.groupby("category")['unit_price'].sum()

In [18]:
# Total revenue per category
revenue_by_cat = df.groupby("category")["revenue"].sum().sort_values(ascending=False)
print(revenue_by_cat)

category
Electronics    12082617.0
Clothing        1321463.0
Healthcare       648889.0
Books            423762.0
Food             163588.0
Name: revenue, dtype: float64


In [26]:
# Multiple aggregations in one pass
df.groupby("category")["revenue"].agg(["sum","mean","count","std"]).round(0)

,sum,mean,count,std
category,,,,
Books,423762.0,3363.0,126,1835.0
Clothing,1321463.0,10657.0,124,6314.0
Electronics,12082617.0,95894.0,126,52906.0
Food,163588.0,1398.0,117,743.0
Healthcare,648889.0,6064.0,107,3489.0


In [27]:
# Named aggregation syntax - cleaner column names
df.groupby("category").agg(
    total_revenue = ("revenue", "sum"),
    avg_order     = ("revenue", "mean"),
    order_count   = ("order_id", "count"),
    avg_discount  = ("discount_pct", "mean"),
).round(1).sort_values("total_revenue", ascending=False)

,total_revenue,avg_order,order_count,avg_discount
category,,,,
Electronics,12082617.0,95893.8,126,4.3
Clothing,1321463.0,10657.0,124,4.6
Healthcare,648889.0,6064.4,107,5.1
Books,423762.0,3363.2,126,4.7
Food,163588.0,1398.2,117,5.7


## Multi-key GroupBy

Pass a list of column names to group on two dimensions at once.

In [28]:
# Revenue by region AND category
region_cat = df.groupby(["region","category"])["revenue"].sum().round(0)
print(region_cat)

region  category   
East    Books            85418.0
        Clothing        348264.0
        Electronics    3618433.0
        Food             28854.0
        Healthcare      125860.0
North   Books           100104.0
        Clothing        377688.0
        Electronics    3760216.0
        Food             47539.0
        Healthcare      159319.0
South   Books           116378.0
        Clothing        333649.0
        Electronics    2316330.0
        Food             41956.0
        Healthcare      199878.0
West    Books           121862.0
        Clothing        261862.0
        Electronics    2387638.0
        Food             45239.0
        Healthcare      163832.0
Name: revenue, dtype: float64


In [31]:
# Unstack the inner key into columns - instantly readable as a heatmap-style table
region_cat.unstack(fill_value=0)

category,Books,Clothing,Electronics,Food,Healthcare
region,,,,,
East,85418.0,348264.0,3618433.0,28854.0,125860.0
North,100104.0,377688.0,3760216.0,47539.0,159319.0
South,116378.0,333649.0,2316330.0,41956.0,199878.0
West,121862.0,261862.0,2387638.0,45239.0,163832.0


## Pivot Tables

`pd.pivot_table()` is syntactic sugar over groupby + unstack. It's particularly useful when you already know the shape you want.

In [34]:
# Revenue by region (rows) x category (columns)
pivot = pd.pivot_table(
    df,
    values  = "revenue",
    index   = "region",
    columns = "category",
    aggfunc = "sum",
    fill_value = 0,
    margins = True,   # adds row/column totals
    margins_name = "TOTAL",
)
pivot.round(0)

category,Books,Clothing,Electronics,Food,Healthcare,TOTAL
region,,,,,,
East,85418.0,348264.0,3618433.0,28854.0,125860.0,4206829.0
North,100104.0,377688.0,3760216.0,47539.0,159319.0,4444866.0
South,116378.0,333649.0,2316330.0,41956.0,199878.0,3008191.0
West,121862.0,261862.0,2387638.0,45239.0,163832.0,2980433.0
TOTAL,423762.0,1321463.0,12082617.0,163588.0,648889.0,14640319.0


In [35]:
# Average discount % by city x category
pd.pivot_table(
    df,
    values  = "discount_pct",
    index   = "city",
    columns = "category",
    aggfunc = "mean",
    fill_value = 0,
).round(1).sort_index()

category,Books,Clothing,Electronics,Food,Healthcare
city,,,,,
Faisalabad,3.3,3.9,4.8,5.6,5.5
Hyderabad,5.8,5.3,5.0,5.3,4.7
Islamabad,5.0,5.2,4.4,4.7,3.8
Karachi,5.0,5.4,5.4,3.8,5.9
Lahore,5.9,4.7,2.9,5.3,4.5
Peshawar,4.8,4.0,3.5,5.9,6.1
Quetta,2.2,6.2,4.2,9.6,4.7
Rawalpindi,5.7,2.3,4.4,5.5,5.0


## .agg() vs .transform()

This is the distinction that trips everyone up the first time.

| | `.agg()` | `.transform()` |
|---|---|---|
| Output shape | One row **per group** | Same shape as **original** df |
| Use case | Summary table | Add a group-level value back to each row |

Think of it this way: `.agg()` **collapses** the group into one number.
`.transform()` **broadcasts** that number back to every row in the group.

In [36]:
# .agg() - collapses to one row per category
cat_revenue_agg = df.groupby("category")["revenue"].agg("sum")
print(f"Shape after .agg(): {cat_revenue_agg.shape}")
print(cat_revenue_agg)

Shape after .agg(): (5,)
category
Books            423762.0
Clothing        1321463.0
Electronics    12082617.0
Food             163588.0
Healthcare       648889.0
Name: revenue, dtype: float64


In [37]:
# .transform() - returns a value for EVERY row, not per group
# Use case: add a 'category total' column so each row knows its group's total
df["category_total"] = df.groupby("category")["revenue"].transform("sum")

# Now we can compute % contribution without a merge
df["pct_of_category"] = (df["revenue"] / df["category_total"] * 100).round(2)

df[["order_id","category","revenue","category_total","pct_of_category"]].head(10)

,order_id,category,revenue,category_total,pct_of_category
0,ORD10210,Clothing,5125.0,1321463.0,0.39
1,ORD10496,Electronics,139484.0,12082617.0,1.15
2,ORD10173,Clothing,6895.0,1321463.0,0.52
3,ORD10081,Books,2446.0,423762.0,0.58
4,ORD10240,Clothing,14752.0,1321463.0,1.12
5,ORD10519,Books,3402.0,423762.0,0.80
6,ORD10534,Books,5555.0,423762.0,1.31
7,ORD10384,Books,3222.0,423762.0,0.76
8,ORD10101,Clothing,12828.0,1321463.0,0.97
9,ORD10491,Books,2700.0,423762.0,0.64


In [38]:
# Verify: transform preserves the original shape
print(f"Original df rows: {len(df)}")
print(f"After adding transform column: {len(df)}")
print(f"Shape unchanged: {df.shape}")

Original df rows: 600
After adding transform column: 600
Shape unchanged: (600, 13)


## Practical

### Practical 1 - groupby().agg() returns a DataFrame, not a Series

In [39]:
# Predict the type and shape BEFORE running
result_series = df.groupby("category")["revenue"].sum()
result_df     = df.groupby("category")[["revenue"]].sum()

print(f"Single column, no brackets: {type(result_series).__name__}  shape {result_series.shape}")
print(f"Single column, brackets:    {type(result_df).__name__}  shape {result_df.shape}")
print()
print("When you chain .agg([...]) you always get a DataFrame.")
print("When you call a single method like .sum() on a Series groupby, you get a Series.")
print("Know which you have - it affects how you access and display results.")

Single column, no brackets: Series  shape (5,)
Single column, brackets:    DataFrame  shape (5, 1)

When you chain .agg([...]) you always get a DataFrame.
When you call a single method like .sum() on a Series groupby, you get a Series.
Know which you have - it affects how you access and display results.


### Practical 2 - nlargest() vs sort_values().head(): same result, very different speed at scale

In [40]:
# Both give the top 3 products by total revenue
by_product = df.groupby("product")["revenue"].sum()

# Method 1: sort everything, then slice
slow_way  = by_product.sort_values(ascending=False).head(3)

# Method 2: nlargest() uses a heap - O(n log k) vs O(n log n) for full sort
fast_way  = by_product.nlargest(3)

print("Top 3 products by revenue:")
print(fast_way.round(0))
print(f"\nSame result: {slow_way.equals(fast_way)}")
print()
print("For small datasets the difference is negligible.")
print("On 10M+ rows pulling top-10 from 100k products: nlargest() wins clearly.")

Top 3 products by revenue:
product
Tablet     3672750.0
Charger    3005050.0
Phone      2467766.0
Name: revenue, dtype: float64

Same result: True

For small datasets the difference is negligible.
On 10M+ rows pulling top-10 from 100k products: nlargest() wins clearly.


### Practical 3 - transform() doesn't change row count - a live proof

In [41]:
original_shape = df.shape

# Add THREE transform-derived columns
df["region_avg_order"] = df.groupby("region")["revenue"].transform("mean")
df["region_order_rank"] = df.groupby("region")["revenue"].transform("rank",
                                                                      ascending=False)
df["above_cat_mean"] = df["revenue"] > df.groupby("category")["revenue"].transform("mean")

print(f"Shape before transforms: {original_shape}")
print(f"Shape after 3 transforms: {df.shape}  <- same row count, 3 new columns")
print()
print("Each new column has the GROUP-LEVEL value broadcast to EVERY row in that group:")
df[["order_id","region","revenue","region_avg_order","above_cat_mean"]].head(8)

Shape before transforms: (600, 13)
Shape after 3 transforms: (600, 16)  <- same row count, 3 new columns

Each new column has the GROUP-LEVEL value broadcast to EVERY row in that group:


,order_id,region,revenue,region_avg_order,above_cat_mean
0,ORD10210,East,5125.0,30048.778571,False
1,ORD10496,East,139484.0,30048.778571,True
2,ORD10173,North,6895.0,27269.116564,False
3,ORD10081,West,2446.0,20002.906040,False
4,ORD10240,North,14752.0,27269.116564,True
5,ORD10519,West,3402.0,20002.906040,True
6,ORD10534,West,5555.0,20002.906040,True
7,ORD10384,South,3222.0,20325.614865,False


### Practical 4 - margins=True in pivot_table: instant row and column totals

In [42]:
# The margins row/column is often the most useful part
revenue_pivot = pd.pivot_table(
    df,
    values     = "revenue",
    index      = "region",
    columns    = "category",
    aggfunc    = "sum",
    fill_value = 0,
    margins    = True,
    margins_name = "TOTAL",
).round(0)

print(revenue_pivot)
print()
print("The 'TOTAL' row and column are not extra work - one parameter adds them.")
print("You can read off the top category (TOTAL column) and top region (TOTAL row)")
print("without any additional computation.")

category     Books   Clothing  Electronics      Food  Healthcare       TOTAL
region                                                                      
East       85418.0   348264.0    3618433.0   28854.0    125860.0   4206829.0
North     100104.0   377688.0    3760216.0   47539.0    159319.0   4444866.0
South     116378.0   333649.0    2316330.0   41956.0    199878.0   3008191.0
West      121862.0   261862.0    2387638.0   45239.0    163832.0   2980433.0
TOTAL     423762.0  1321463.0   12082617.0  163588.0    648889.0  14640319.0

The 'TOTAL' row and column are not extra work - one parameter adds them.
You can read off the top category (TOTAL column) and top region (TOTAL row)
without any additional computation.


## Practical Exercise - E-commerce Sales Analytics
**Difficulty: Medium**

In [43]:
# Task 1: Which 3 products generated the most revenue? Show product, category, total.
top_products = (df.groupby(["product","category"])["revenue"]
                  .sum()
                  .nlargest(3)
                  .reset_index())
top_products.columns = ["product","category","total_revenue"]
top_products["total_revenue"] = top_products["total_revenue"].round(0)
top_products

,product,category,total_revenue
0,Tablet,Electronics,3672750.0
1,Charger,Electronics,3005050.0
2,Phone,Electronics,2467766.0


In [44]:
# Task 2: Average order value and count by region, sorted by avg descending
df.groupby("region").agg(
    orders          = ("order_id","count"),
    total_revenue   = ("revenue","sum"),
    avg_order_value = ("revenue","mean"),
    avg_discount    = ("discount_pct","mean"),
).round(1).sort_values("avg_order_value", ascending=False)

,orders,total_revenue,avg_order_value,avg_discount
region,,,,
East,140,4206829.0,30048.8,4.5
North,163,4444866.0,27269.1,4.7
South,148,3008191.0,20325.6,5.2
West,149,2980433.0,20002.9,5.1


In [45]:
# Task 3: Revenue pivot - region x month
df["month"] = df["date"].dt.to_period("M").astype(str)

monthly_pivot = pd.pivot_table(
    df,
    values     = "revenue",
    index      = "region",
    columns    = "month",
    aggfunc    = "sum",
    fill_value = 0,
    margins    = True,
    margins_name = "TOTAL",
).round(0)

# Show first 6 months + TOTAL column to keep it readable
cols_to_show = [c for c in monthly_pivot.columns if c <= "2023-06" or c == "TOTAL"]
monthly_pivot[cols_to_show]

month,2023-01,2023-02,2023-03,2023-04,2023-05,2023-06,TOTAL
region,,,,,,,
East,443253.0,612995.0,430353.0,848025.0,396483.0,568115.0,4206829.0
North,231312.0,815358.0,177033.0,290281.0,452428.0,446433.0,4444866.0
South,302629.0,145779.0,417698.0,75664.0,48482.0,88005.0,3008191.0
West,367718.0,97459.0,90594.0,306865.0,277159.0,590149.0,2980433.0
TOTAL,1344912.0,1671591.0,1115678.0,1520835.0,1174552.0,1692702.0,14640319.0


In [46]:
# Task 4: pct_of_category already added above - verify it sums to ~100% per category
check = df.groupby("category")["pct_of_category"].sum().round(1)
print("Sum of pct_of_category per category (should be 100.0):")
print(check)

Sum of pct_of_category per category (should be 100.0):
category
Books          100.0
Clothing       100.0
Electronics    100.0
Food           100.0
Healthcare     100.0
Name: pct_of_category, dtype: float64


## Summary

| Concept | Key point |
|---|---|
| Split-Apply-Combine | The mental model behind every `.groupby()` call |
| `.agg(name=(col, func))` | Named aggregation - clean output column names |
| Multi-key groupby | Pass a list: `.groupby(["col1","col2"])` |
| `.unstack()` | Pivots the inner group key into columns |
| `pd.pivot_table(margins=True)` | Adds row/column totals automatically |
| `.agg()` vs `.transform()` | agg collapses to per-group; transform broadcasts back to every row |

## Homework

Predict: if you `.groupby("category")["revenue"].transform("rank")` - what does the
result look like? How many rows will it have? Will the numbers be 1,2,3... or
something else? Test your prediction tomorrow.

**Summary Table**

| Method / Parameter | Output Shape | Main Use Case | Key Advantages / Notes |
| --- | --- | --- | --- |
| **`df.groupby('key')['col'].sum()`** | Series ($G$ rows) | Single-metric group summary | Returns a Pandas Series indexed by the group key.|
| **`df.groupby('key')[['col']].sum()`** | DataFrame ($G \times 1$) | Single-metric group summary | Using double brackets `[['col']]` forces a DataFrame output.|
| **`.agg(col_name=('target', 'func'))`** | DataFrame ($G \times M$) | Custom multi-metric summary | Named tuple syntax prevents messy MultiIndex columns.|
| **`.unstack()`** | DataFrame | Index-to-column reshaping | Pivots the innermost MultiIndex level into horizontal columns.|
| **`pd.pivot_table(..., margins=True)`** | DataFrame | 2D Cross-tabulation | `margins=True` computes row/column totals without extra joins.|
| **`df.groupby('key')['col'].transform('func')`** | Series ($N$ rows) | Group broadcasting | Keeps original row count $N$; computes relative % or z-scores.|
| **`Series.nlargest(k)`** | Series ($k$ rows) | Top-$k$ filtering | $O(N \log k)$ complexity; faster than `.sort_values().head(k)`.|


**Question 1: Shape & Return Type Identification**

* *"What is the structural difference between `df.groupby('category')['revenue'].sum()` and `df.groupby('category')[['revenue']].sum()`? Why does this distinction matter when chaining downstream methods?"*

**Question 2: The `.transform()` Shape Invariant**
* *"If `df` has 600 rows across 5 categories, what will be the shape of `df.groupby('category')['revenue'].transform('mean')`? Can you pass this result directly into `df['category_avg'] = ...`?"*
  
**Question 3: MultiIndex Flattening**
* *"When you run `df.groupby(['region', 'category'])['revenue'].sum()`, what does the index look like? How do you convert it back into a flat table with standard columns?"*

**Question 4: `nlargest()` Performance Mechanics**
* *"Why is `df.groupby('product')['revenue'].sum().nlargest(3)` computationally superior to `.sort_values(ascending=False).head(3)` on a dataset with 10 million rows?"*

**Question:** The marketing team wants to identify `"VIP Customers"` for a loyalty campaign. A VIP is `defined by two metrics`: how many `separate orders` they placed, and their `total lifetime revenue`.


Write a Pandas operation `grouping by customer_id` that `calculates both the total revenue` and `the distinct number of orders per customer`, `renaming` the columns to `lifetime_value and order_count`. How would you isolate the top 5 customers by lifetime_value?

In [52]:
#use named aggregation to create clean columns in one pass
vip_customers = df.groupby("customer_id").agg(
    lifetime_value = ("revenue", "sum"),
    order_count    = ("order_id", "nunique") # nunique counts distinct orders
)

#fetch top 5 using nlargest for optimal performance
top_5_vips = vip_customers.nlargest(5, "lifetime_value")
print(top_5_vips)

             lifetime_value  order_count
customer_id                             
C1276              333989.0            3
C1938              326200.0            4
C1433              274642.0            3
C1188              243282.0            4
C1794              240319.0            3
